In [1]:
import os 
import json
import random
import pickle as pk
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from typing import List
from typing import Literal
import base64
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

True

In [3]:
ASSOCIATION_DIR = os.getenv("ASSOCIATION_DIR")
TABLE_DIR = os.getenv("TABLE_DIR")
OPENAI_KEY = os.getenv("OPENAI_KEY")
FINAL_DATASET_IMAGES = os.getenv("FINAL_DATASET_IMAGES")
CRITERIA_EXTRACTION_DIR = os.getenv("CRITERIA_EXTRACTION_DIR")
QUESTIONS_MULTIMODALQA_TRAINING = os.getenv("QUESTIONS_MULTIMODALQA_TRAINING")
selected_questions = pk.load(open("/Users/emanuelemezzi/PycharmProjects/LEGuidance/scripts/selected_questions.pk", "rb"))

In [4]:
selected_questions

['question_3158.json',
 'question_13163.json',
 'question_10777.json',
 'question_6283.json',
 'question_17285.json',
 'question_16529.json',
 'question_16079.json',
 'question_13905.json',
 'question_6182.json',
 'question_12981.json',
 'question_4386.json',
 'question_6267.json',
 'question_22612.json',
 'question_12708.json',
 'question_23299.json',
 'question_19947.json',
 'question_11084.json',
 'question_11050.json',
 'question_12782.json',
 'question_15982.json',
 'question_16292.json',
 'question_21505.json',
 'question_23629.json',
 'question_11031.json',
 'question_22880.json',
 'question_15684.json',
 'question_5474.json',
 'question_3396.json',
 'question_4979.json',
 'question_9685.json',
 'question_19913.json',
 'question_16006.json',
 'question_20380.json',
 'question_23275.json',
 'question_17301.json',
 'question_7986.json',
 'question_10149.json',
 'question_6683.json',
 'question_7864.json',
 'question_14480.json',
 'question_493.json',
 'question_964.json',
 'questi

In [5]:
dataset = pk.load(open("tree_dataset.pkl", "rb"))

In [6]:
dataset

,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality
question_3158.json,Did Cristiano dos Santos Rodrigues play in mor...,76,Sports,season (which had more games played),comparison result,"[table, table]"
question_13163.json,Which of the home media of ICarly : the Comple...,99,Television,home media release (which one has higher disc ...,media release,"[table, table]"
question_10777.json,"Was It's Like, You Know... or Friends, the one...",106,Television,television series title (one of two options),work,"[table, table]"
question_6283.json,The Usher song Yeah! charted lower in which Ch...,119,Music,chart (music chart name/market),chart,"[table, table]"
question_17285.json,In the chart performance from 1983 for the son...,152,Music,chart name (which had the lower peak position),string,"[table, table]"
...,...,...,...,...,...,...
question_23543.json,"Which Location(s), in Teams and locations of 1...",112,Sports,location(s) (place names listed in 'Teams and ...,location,[image]
question_15516.json,What color jacket is Hammer & Tongs wearing?,44,Other,color (jacket color),attribute,[image]
question_189.json,What color teeth does Goldie have?,34,Other,color (teeth color),attribute,[image]
question_14280.json,"Is the red stripe on the left, center, or righ...",76,Geography,relative position (left/center/right),attribute,[image]


In [178]:
def dataset_build(dataset):
    image_rows = dataset[dataset["modality"].astype(str) == "['image']"].sample(20, random_state=42)
    text_rows  = dataset[dataset["modality"].astype(str) == "['text']"].sample(20, random_state=42)
    table_rows = dataset[dataset["modality"].astype(str) == "['table']"].sample(20, random_state=42)
    
    test_dataset = pd.concat([image_rows, text_rows, table_rows]) \
                   .sample(frac=1, random_state=42) \
                   .reset_index(drop=False)
    
    return test_dataset

In [179]:
test_dataset = dataset_build(dataset)

In [180]:
test_dataset["modality_answers"] = [[] for _ in range(len(test_dataset))]

In [181]:
test_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_13547.json,What object is seen in the center of the Days ...,66,Television,object depicted (center of poster),object,[image],[]
1,question_14691.json,How many people are wearing a black and red st...,92,Sports,count of people,number,[image],[]
2,question_5497.json,The Red Bull RB8 is a Formula One racing car d...,240,Sports,racing team / organization,organization,[text],[]
3,question_11881.json,"was Tha Eastsidaz the Featuring, in track list...",106,Music,yes/no (whether an artist is a featured act on...,boolean,[table],[]
4,question_10110.json,What general pattern is on Lionel Messi's jersey?,49,Sports,jersey pattern (description),attribute,[image],[]
5,question_5821.json,Was the 2005-06 AHL season in Scott Gordon's m...,96,Sports,yes/no (boolean),boolean,[table],[]
6,question_10556.json,who plays nina's mom on general hospital,40,Television,actor/actress (cast member),person,[text],[]
7,question_11778.json,Who was the operator when the route was N30 in...,72,Transportation,operator (organization/company),organization,[table],[]
8,question_11017.json,What color is the car parked up against the cu...,119,Other,color,attribute,[image],[]
9,question_10128.json,In what year was Lauren Crace in the film Silk?,47,Film,year,time,[table],[]


In [41]:
class ModalityResponse(BaseModel):
    answer: Literal["YES", "NO"] = Field(..., description="Whether the question can be answered using ONLY the given data element and its metadata")

In [14]:
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [15]:
client = OpenAI(api_key=OPENAI_KEY)

# First Approach: It also tells which specific piece of data contains the answer

In [95]:
def prompt_image(question, image): 
    image64 = encode_image(os.path.join(FINAL_DATASET_IMAGES, image["path"]))
    
    print("Question: ", question)
    print("Image title: ", image["title"])
    print("Image path: ", image["path"])

    response = client.responses.parse(
        model="gpt-5.2",
        input=[
            {
                "role": "system",
                "content": """You are a modality suitability judge.
                            
                You will be given:
                - A question
                - ONE image
                - The title of the image
                
                Your job is to decide whether the question can be answered using ONLY this single image and its title.
                
                Rules:
                - Use only the provided data item. Do not use outside knowledge or your knowledge.
                - Do not assume other images exist.
                - Do not guess.
                - If the answer is not directly visible or inferable from the image/title, respond NO.
                - If the image/title contains enough information to confidently answer, respond YES.
                
                Output MUST be exactly one word: YES or NO.
                """
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": f"""Question: {question}
                        Title: {image["title"]}

                        Can the question be answered using ONLY this image and its title?
                        """
                    },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{image64}"
                    },
                ]
            }
        ],
        text_format=ModalityResponse,
    )
    
    answer = response.output_parsed.answer
    print(answer)
    return answer

In [96]:
def prompt_text(question, text):
    
    print("Question: ", question)
    print("Title: ", text["title"])
    print("Text: ", text["text"])

    response = client.responses.parse(
        model="gpt-5.2",
        input=[
            {
                "role": "system",
                "content": """You are a modality suitability judge.
                            
                You will be given:
                - A question
                - ONE paragraph (text content)
                - The title of the paragraph
                
                Your job is to decide whether the question can be answered using ONLY this single paragraph and its title.
                
                Rules:
                - Use only the provided paragraph and title. Do not use outside knowledge or your own knoweldge.
                - Do not assume other paragraphs exist.
                - Do not guess.
                - If the answer is not explicitly stated or clearly derivable from the paragraph/title, respond NO.
                - If the paragraph/title contains enough information to confidently answer, respond YES.
                
                Output MUST be exactly one word: YES or NO.
                """
            },
            {
                "role": "user",
                "content": f"""Question: {question}
                
                Title: {text["title"]}
                
                Paragraph Content: {text["text"]}

                Can the question be answered using ONLY this paragraph and its title?
                """
            }
        ],
        text_format=ModalityResponse,
    )
    
    answer = response.output_parsed.answer
    print(answer)
    return answer

In [136]:
def prompt_table(question, table):
    
    json_table = json.load(open(os.path.join(TABLE_DIR, table["json"]), "rb"))
    
    print("Question: ", question)
    print("Title: ", json_table["title"])
    print("Table: ", json_table["table"])

    response = client.responses.parse(
        model="gpt-5.2",
        input=[
            {
                "role": "system",
                "content": """You are a modality suitability judge.

                You will be given:
                - A question
                - ONE table (structured data)
                - The title of the table
                
                Your job is to decide whether the question can be answered using ONLY this single table and its title.
                
                Rules:
                - Use only the provided table and title. Do not use outside knowledge or your knowledge.
                - Do not assume other tables exist.
                - Do not guess.
                - If the answer is not explicitly present or cannot be derived from the table, respond NO.
                - If the table contains enough information to confidently answer, respond YES.
                
                Output MUST be exactly one word: YES or NO.
                """
            },
            {
                "role": "user",
                "content": f"""Question: {question["original_question"]}
                
                Title: {json_table["title"]}
                
                Table Content: {json_table["table"]}
                
                Can the question be answered using ONLY this table and its title?
                """
            }
        ],
        text_format=ModalityResponse,
    )
    
    answer = response.output_parsed.answer
    print(answer)
    return answer

In [98]:
def gather_answers(question, modality, set): 
    answers, yes_dict = [], {"YES": []}
    if modality == "image": 
        for data_sample in set: 
            answer = prompt_image(question["original_question"], data_sample)
            answers.append(answer)
            if answer == "YES": yes_dict["YES"].append(question["index"])
    elif modality == "text": 
        for data_sample in set: 
            answer = prompt_text(question["original_question"], data_sample)
            answers.append(answer)
            if answer == "YES": yes_dict["YES"].append(question["index"])
    elif modality == "table": 
        for data_sample in set: 
            answer = prompt_table(question["original_question"], data_sample)
            answers.append(answer)
            if answer == "YES": yes_dict["YES"].append(question["index"])
            
    return answers

In [99]:
def check_modality(answers, modality):
    if any(answer=="YES" for answer in answers): 
        return modality

In [100]:
for i, question in test_dataset.iterrows(): 
    print(f"Question: {i}, {question['index']}")
    json_association = json.load(open(os.path.join(ASSOCIATION_DIR, question["index"]), "rb"))
    
    image_set = json_association["image_set"][:2]
    text_set = json_association["text_set"][:2]
    table_set = json_association["table_set"][:1]
    
    # Gather answers for image
    answers_image = gather_answers(question, "image", image_set)
    print("The answers for the images are: ", answers_image)
    check_modality(answers_image, "image")
    
    # Gather answers for text
    answers_text = gather_answers(question, "text", text_set)
    check_modality(answers_text, "text")
    
    # Gather answers for table
    answers_table = gather_answers(question, "table", table_set)
    check_modality(answers_table, "table")
    
    break

Question: 0, question_13547.json
Question:  What object is seen in the center of the Days of Our Lives poster?
Image title:  Days of Our Lives
Image path:  02e80d4acbc03c25592eeae21054713d.jpg
YES
Question:  What object is seen in the center of the Days of Our Lives poster?
Image title:  Kickin' It
Image path:  592fac145be775ffcb94cd4a6a69cd0f.png
NO
The answers for the images are:  ['YES', 'NO']
Question:  What object is seen in the center of the Days of Our Lives poster?
Title:  Boy Scout Memorial
Text:  The sculpture consists of three bronze figures: a Boy Scout in the center wearing a uniform stepping forward and carrying a walking stick in his left hand. Flanking him are two larger allegorical figures of a man and woman. They represent "American Manhood and Womanhood and the ideals they will pass onto the youth." To the Boy Scout's right side is the male figure, nearly nude, who carries a bundle of leaves and drapery in his left arm. Part of the drapery blows across his middle as 

# Second Approach: Here we now do that we give all the images at ones, all the paragraphs at ones

In [195]:
class ModalityResponse(BaseModel):
    answer: Literal["YES", "NO"] = Field(..., description="Whether the question can be answered using ONLY the given data elements and their metadata")

In [196]:
def prompt_images(question, images):
    # Encode all images
    image_inputs = []
    for img in images:
        image64 = encode_image(os.path.join(FINAL_DATASET_IMAGES, img["path"]))
        image_inputs.append({
            "title": img["title"],
            "image_url": f"data:image/jpeg;base64,{image64}"
        })
    
    # Prepare textual description of all images
    images_text = "\n".join([f"{i+1}. Title: {img['title']}" for i, img in enumerate(images)])

    #print("Question: ", question["original_question"])
    #print("Images provided:")
    for img in images:
        print("-", img["title"], img["path"])
        
    #print("Images texts: ", images_text)
    
    response = client.responses.parse(
        model="gpt-5.2",
        input=[
            {
                "role": "system",
                "content": """You are a modality suitability judge.

                You will be given:
                - A question
                - A list of images with their titles
                
                Your job is to decide whether the question can be answered using ONLY the information in these images and their titles.
                
                Rules:
                - Use only the provided images and titles. Do not use outside knowledge or your knowledge.
                - Do not guess.
                - If the answer is derivable from at least one image/title, respond YES.
                - If none of the images contain enough information, respond NO.
                
                Output MUST be exactly one word: YES or NO.
                """
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": f"""Question: {question["original_question"]}
                        
                        Images: {images_text}
                        
                        Can the question be answered using ONLY these images and their titles?
                        """
                    },
                    
                    # attach all images
                    *[
                        {"type": "input_image", "image_url": img["image_url"]} for img in image_inputs
                    ]
                ]
            }
        ],
        text_format=ModalityResponse,
    )
    
    answer = response.output_parsed.answer
    return answer

In [225]:
def prompt_texts(question, paragraphs):
    
    # Prepare textual representation for all paragraphs
    paragraphs_text = "\n\n".join([
        f"{i+1}. Title: {p['title']}\nContent: {p['text']}" 
        for i, p in enumerate(paragraphs)
    ])
    
    #print("Question:", question["original_question"])
    #print("Paragraphs provided:")
    for p in paragraphs:
        print("-", p["title"])
        
    print("Paragraphs texts: ", paragraphs_text)
    
    response = client.responses.parse(
        model="gpt-5.2",
        input=[
            {
                "role": "system",
                "content": """You are a modality suitability judge.

                            You will be given:
                            - A question
                            - A list of paragraphs with their titles
                            
                            Your job is to decide whether the question can be answered using ONLY the information in these paragraphs and their titles.
                            
                            Rules:
                            - Use only the provided paragraphs and titles. Do not use outside knowledge.
                            - Do not guess.
                            - If the answer can be derived from at least one paragraph/title, respond YES.
                            - If none of the paragraphs contain enough information, respond NO.
                            
                            Output MUST be exactly one word: YES or NO.
                            """
            },
            {
                "role": "user",
                "content": f"""Question: {question["original_question"]}
                
                Paragraphs: {paragraphs_text}
                
                Can the question be answered using ONLY at least one these paragraphs and their titles?
                """
            }
        ],
        text_format=ModalityResponse,
    )
    
    answer = response.output_parsed.answer
    return answer

In [198]:
def prompt_table(question, table):
    
    json_table = json.load(open(os.path.join(TABLE_DIR, table["json"]), "rb"))
    
    #print("Question: ", question["original_question"])
    #print("Title: ", json_table["title"])
    #print("Table: ", json_table["table"])

    response = client.responses.parse(
        model="gpt-5.2",
        input=[
            {
                "role": "system",
                "content": """You are a modality suitability judge.

                You will be given:
                - A question
                - ONE table (structured data)
                - The title of the table
                
                Your job is to decide whether the question can be answered using ONLY this single table and its title.
                
                Rules:
                - Use only the provided table and title. Do not use outside knowledge or your knowledge.
                - Do not assume other tables exist.
                - Do not guess.
                - If the answer is not explicitly present or cannot be derived from the table, respond NO.
                - If the table contains enough information to confidently answer, respond YES.
                
                Output MUST be exactly one word: YES or NO.
                """
            },
            {
                "role": "user",
                "content": f"""Question: {question["original_question"]}
                
                Title: {json_table["title"]}
                
                Table Content: {json_table["table"]}
                
                Can the question be answered using ONLY this table and its title?
                """
            }
        ],
        text_format=ModalityResponse,
    )
    
    answer = response.output_parsed.answer
    return answer

In [256]:
def derive_answers(dataset): 
    for i, question in dataset.iterrows(): 
        print(f"Question number: {i}, Question JSON: {question['index']}, Question: {question['original_question']}")
        json_association = json.load(open(os.path.join(ASSOCIATION_DIR, question["index"]), "rb"))
        
        image_set = json_association["image_set"]
        text_set = json_association["text_set"]
        table_set = json_association["table_set"][0]
        
        answer_images = prompt_images(question, image_set)
        answer_texts = prompt_texts(question, text_set)
        answer_table = prompt_table(question, table_set)
        
        print("Answer for images: ", answer_images)
        print("Answer for text: ", answer_texts)
        print("Answer for table: ", answer_table)
        
        modality_answers = []
        
        if answer_images == "YES":
            modality_answers.append("image")
            
        if answer_texts == "YES": 
            modality_answers.append("text")
            
        if answer_table == "YES": 
            modality_answers.append("table")
        
        dataset.at[i, "modality_answers"] = modality_answers

In [232]:
test_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_13547.json,What object is seen in the center of the Days ...,66,Television,object depicted (center of poster),object,[image],[image]
1,question_14691.json,How many people are wearing a black and red st...,92,Sports,count of people,number,[image],[image]
2,question_5497.json,The Red Bull RB8 is a Formula One racing car d...,240,Sports,racing team / organization,organization,[text],[text]
3,question_11881.json,"was Tha Eastsidaz the Featuring, in track list...",106,Music,yes/no (whether an artist is a featured act on...,boolean,[table],[]
4,question_10110.json,What general pattern is on Lionel Messi's jersey?,49,Sports,jersey pattern (description),attribute,[image],[image]
5,question_5821.json,Was the 2005-06 AHL season in Scott Gordon's m...,96,Sports,yes/no (boolean),boolean,[table],[]
6,question_10556.json,who plays nina's mom on general hospital,40,Television,actor/actress (cast member),person,[text],[text]
7,question_11778.json,Who was the operator when the route was N30 in...,72,Transportation,operator (organization/company),organization,[table],[table]
8,question_11017.json,What color is the car parked up against the cu...,119,Other,color,attribute,[image],[image]
9,question_10128.json,In what year was Lauren Crace in the film Silk?,47,Film,year,time,[table],[table]


In [252]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

In [254]:
def calculate_metrics(dataset): 
    mlb = MultiLabelBinarizer()
    
    # Fit and transform both ground truth and predictions
    y_true = mlb.fit_transform(dataset["modality"])
    y_pred = mlb.transform(dataset["modality_answers"])
    
    precision_per_modality = precision_score(y_true, y_pred, average=None, zero_division=0)
    recall_per_modality = recall_score(y_true, y_pred, average=None, zero_division=0)
    
    print("Modalities:", mlb.classes_)
    print("Precision per modality:", precision_per_modality)
    print("Recall per modality:", recall_per_modality)

In [240]:
pk.dump(test_dataset, open("unimodal_multiple_prompts.pk", "wb"))

# Third approach: here we consider also the multimodal case

In [52]:
def dataset_build(dataset):
    image_text_rows = dataset[dataset["modality"].astype(str) == "['image', 'text']"].sample(5, random_state=42)
    text_image_rows = dataset[dataset["modality"].astype(str) == "['text', 'image']"].sample(5, random_state=42)
    image_table_rows  = dataset[dataset["modality"].astype(str) == "['image', 'table']"].sample(5, random_state=42)
    table_image_rows  = dataset[dataset["modality"].astype(str) == "['table', 'image']"].sample(5, random_state=42)
    text_table_rows  = dataset[dataset["modality"].astype(str) == "['text', 'table']"].sample(5, random_state=42)
    table_text_rows  = dataset[dataset["modality"].astype(str) == "['table', 'text']"].sample(5, random_state=42)
    table_table_rows  = dataset[dataset["modality"].astype(str) == "['table', 'table']"].sample(5, random_state=42)
    
    test_dataset = pd.concat([image_text_rows, text_image_rows, image_table_rows, table_image_rows, text_table_rows, table_text_rows, table_table_rows]) \
                   .sample(frac=1, random_state=42) \
                   .reset_index(drop=False)
    
    return test_dataset

In [53]:
multimodal_dataset = dataset_build(dataset)

In [54]:
multimodal_dataset["modality_answers"] = [[] for _ in range(len(multimodal_dataset))]

In [55]:
multimodal_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_1269.json,Who had a hit with the artist of the music per...,160,Television,person or group (hitmaking artist),person,"[table, text]",[]
1,question_19018.json,Between the driver that is sponsored by a comp...,174,Sports,driver (person/competitor name),person,"[image, table]",[]
2,question_22384.json,At Soldier Field the Chicago Bears played agai...,150,Sports,NFL team names,sports teams,"[text, table]",[]
3,question_6153.json,Was Indiana University or the school where sci...,163,Sports,choice between two entities (yes/no comparison...,comparison outcome,"[text, table]",[]
4,question_15310.json,The venue where Sian Brooke starred in Wanderl...,78,Television,number of cars,number,"[table, image]",[]
5,question_6883.json,Does Hilary Duff sing in the movie where T.J. ...,69,Film,yes/no (whether Hilary Duff sings in a specifi...,boolean,"[table, text]",[]
6,question_2776.json,On the poster for the television show that Cha...,103,Television,time of day (as shown on a poster),time,"[table, image]",[]
7,question_9578.json,Which film title did Dick Wesson act in first:...,108,Film,film title,creative work,"[image, table]",[]
8,question_20579.json,"in Team Performance of World Club Challenge, w...",144,Sports,team (club) name,sports team,"[text, image]",[]
9,question_6113.json,What vehicle is Crash Bandicoot on the cover o...,84,Video games,vehicle (type/name shown on a cover),vehicle,"[table, image]",[]


In [11]:
derive_answers(multimodal_dataset)

NameError: name 'derive_answers' is not defined

# Fifth Approach: Multimodal with only one prompt

In [56]:
class ModalityDecision(BaseModel):
    modalities: Literal[
        "image_text",
        "image_table",
        "text_table",
        "table_table"
    ] = Field(..., description="Predicted modality combination")

In [80]:
def prompt_multimodal(question, images, paragraphs, table):
    
    images_text = "\n\n".join([
        f"{i+1}. Title: {img['title']}" 
        for i, img in enumerate(images)
    ])

    image_inputs = []
    for img in images:
        image64 = encode_image(os.path.join(FINAL_DATASET_IMAGES, img["path"]))
        image_inputs.append({
            "title": img["title"],
            "image_url": f"data:image/jpeg;base64,{image64}"
        })
        
    paragraphs_text = "\n\n".join([
        f"{i+1}. Title: {p['title']}\nContent: {p['text']}" 
        for i, p in enumerate(paragraphs)
    ])
    
    json_table = json.load(open(os.path.join(TABLE_DIR, table["json"]), "rb"))

    tables_text = f"""
    Title: {json_table["title"]}
    Content: {json_table["table"]}
    """

    response = client.responses.parse(
        model="gpt-5.2",
        input=[
            {
                "role": "system",
                "content": """
You are a multimodal modality-selection classifier.

You will be given:
- A question
- IMAGES (titles + image content)
- TEXT paragraphs (titles + content)
- TABLES (titles + table content)

Your job is to decide which modality combination is required to answer the question.

IMPORTANT:
The output must be EXACTLY ONE of the following labels:

- image_text  (answer requires combining IMAGE + TEXT)
- image_table  (answer requires combining IMAGE + TABLE)
- text_table   (answer requires combining TEXT + TABLE)
- table_table  (answer requires combining TWO ROWS FROM THE SAME TABLE)

Rules:
- Use ONLY the provided data.
- Do NOT use outside knowledge or your knowledge.
- Do NOT guess.
- Pick the best matching combination that makes the question answerable.
- Output ONLY the label (no explanation).
"""
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": f"""
Question:
{question["original_question"]}

IMAGES:
{images_text}

TEXT paragraphs:
{paragraphs_text}

TABLES:
{tables_text}

Which modality combination is required? Return only one label.
"""
                    },
                    *[
                        {"type": "input_image", "image_url": img["image_url"]}
                        for img in image_inputs
                    ]
                ]
            }
        ],
        text_format=ModalityDecision,
    )

    return response.output_parsed.modalities

In [58]:
multimodal_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_1269.json,Who had a hit with the artist of the music per...,160,Television,person or group (hitmaking artist),person,"[table, text]",[]
1,question_19018.json,Between the driver that is sponsored by a comp...,174,Sports,driver (person/competitor name),person,"[image, table]",[]
2,question_22384.json,At Soldier Field the Chicago Bears played agai...,150,Sports,NFL team names,sports teams,"[text, table]",[]
3,question_6153.json,Was Indiana University or the school where sci...,163,Sports,choice between two entities (yes/no comparison...,comparison outcome,"[text, table]",[]
4,question_15310.json,The venue where Sian Brooke starred in Wanderl...,78,Television,number of cars,number,"[table, image]",[]
5,question_6883.json,Does Hilary Duff sing in the movie where T.J. ...,69,Film,yes/no (whether Hilary Duff sings in a specifi...,boolean,"[table, text]",[]
6,question_2776.json,On the poster for the television show that Cha...,103,Television,time of day (as shown on a poster),time,"[table, image]",[]
7,question_9578.json,Which film title did Dick Wesson act in first:...,108,Film,film title,creative work,"[image, table]",[]
8,question_20579.json,"in Team Performance of World Club Challenge, w...",144,Sports,team (club) name,sports team,"[text, image]",[]
9,question_6113.json,What vehicle is Crash Bandicoot on the cover o...,84,Video games,vehicle (type/name shown on a cover),vehicle,"[table, image]",[]


In [81]:
def derive_answers_multimodal_one_prompt(dataset): 
    for i, question in dataset.iterrows(): 
        print(f"Question number: {i}, Question JSON: {question['index']}, Question: {question['original_question']}")
        json_association = json.load(open(os.path.join(ASSOCIATION_DIR, question["index"]), "rb"))
        
        image_set = json_association["image_set"]
        text_set = json_association["text_set"]
        table_set = json_association["table_set"][0]
        
        modality_answer = prompt_multimodal(question, image_set, text_set, table_set)
        
        print("Answer for images: ", modality_answer)
        
        dataset.at[i, "modality_answers"] = modality_answer

In [60]:
derive_answers_multimodal_one_prompt(multimodal_dataset)

Question number: 0, Question JSON: question_1269.json, Question: Who had a hit with the artist of the music performed during Magic Moments of Week 7 of season 6 of the German Let's dance when the score was 16 (7,5,4) in 2012?
Answer for images:  text_table
Question number: 1, Question JSON: question_19018.json, Question: Between the driver that is sponsored by a company with a yellow triangle logo below its name and Matt Kenseth, which driver had the lowest position in the 2014 Brickyard 400?
Answer for images:  image_table
Question number: 2, Question JSON: question_22384.json, Question: At Soldier Field the Chicago Bears played against what teams that have won the most Super Bowls in a row during their regular season schedule of 1995?
Answer for images:  text_table
Question number: 3, Question JSON: question_6153.json, Question: Was Indiana University or the school where scientists built the first nanotube computer associated with a lower pick in the first round of the 2011 MLS Super

In [61]:
multimodal_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_1269.json,Who had a hit with the artist of the music per...,160,Television,person or group (hitmaking artist),person,"[table, text]",text_table
1,question_19018.json,Between the driver that is sponsored by a comp...,174,Sports,driver (person/competitor name),person,"[image, table]",image_table
2,question_22384.json,At Soldier Field the Chicago Bears played agai...,150,Sports,NFL team names,sports teams,"[text, table]",text_table
3,question_6153.json,Was Indiana University or the school where sci...,163,Sports,choice between two entities (yes/no comparison...,comparison outcome,"[text, table]",text_table
4,question_15310.json,The venue where Sian Brooke starred in Wanderl...,78,Television,number of cars,number,"[table, image]",image_table
5,question_6883.json,Does Hilary Duff sing in the movie where T.J. ...,69,Film,yes/no (whether Hilary Duff sings in a specifi...,boolean,"[table, text]",text_table
6,question_2776.json,On the poster for the television show that Cha...,103,Television,time of day (as shown on a poster),time,"[table, image]",image_table
7,question_9578.json,Which film title did Dick Wesson act in first:...,108,Film,film title,creative work,"[image, table]",image_table
8,question_20579.json,"in Team Performance of World Club Challenge, w...",144,Sports,team (club) name,sports team,"[text, image]",image_table
9,question_6113.json,What vehicle is Crash Bandicoot on the cover o...,84,Video games,vehicle (type/name shown on a cover),vehicle,"[table, image]",image_table


In [62]:
pk.dump(multimodal_dataset, open("multimodal_one_prompt.pk", "wb"))

In [44]:
multimodal_dataset = pk.load(open("multimodal_one_prompt.pk", "rb"))

In [63]:
multimodal_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_1269.json,Who had a hit with the artist of the music per...,160,Television,person or group (hitmaking artist),person,"[table, text]",text_table
1,question_19018.json,Between the driver that is sponsored by a comp...,174,Sports,driver (person/competitor name),person,"[image, table]",image_table
2,question_22384.json,At Soldier Field the Chicago Bears played agai...,150,Sports,NFL team names,sports teams,"[text, table]",text_table
3,question_6153.json,Was Indiana University or the school where sci...,163,Sports,choice between two entities (yes/no comparison...,comparison outcome,"[text, table]",text_table
4,question_15310.json,The venue where Sian Brooke starred in Wanderl...,78,Television,number of cars,number,"[table, image]",image_table
5,question_6883.json,Does Hilary Duff sing in the movie where T.J. ...,69,Film,yes/no (whether Hilary Duff sings in a specifi...,boolean,"[table, text]",text_table
6,question_2776.json,On the poster for the television show that Cha...,103,Television,time of day (as shown on a poster),time,"[table, image]",image_table
7,question_9578.json,Which film title did Dick Wesson act in first:...,108,Film,film title,creative work,"[image, table]",image_table
8,question_20579.json,"in Team Performance of World Club Challenge, w...",144,Sports,team (club) name,sports team,"[text, image]",image_table
9,question_6113.json,What vehicle is Crash Bandicoot on the cover o...,84,Video games,vehicle (type/name shown on a cover),vehicle,"[table, image]",image_table


In [22]:
def normalize_combo(x):
    if isinstance(x, list): 
        return "_".join(sorted(x)) 
    elif isinstance(x, str): 
        return '_'.join(sorted(x.split('_')))

In [65]:
multimodal_dataset["modality"] = multimodal_dataset["modality"].apply(normalize_combo)
multimodal_dataset["modality_answers"] = multimodal_dataset["modality_answers"].apply(normalize_combo)

In [66]:
multimodal_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_1269.json,Who had a hit with the artist of the music per...,160,Television,person or group (hitmaking artist),person,table_text,table_text
1,question_19018.json,Between the driver that is sponsored by a comp...,174,Sports,driver (person/competitor name),person,image_table,image_table
2,question_22384.json,At Soldier Field the Chicago Bears played agai...,150,Sports,NFL team names,sports teams,table_text,table_text
3,question_6153.json,Was Indiana University or the school where sci...,163,Sports,choice between two entities (yes/no comparison...,comparison outcome,table_text,table_text
4,question_15310.json,The venue where Sian Brooke starred in Wanderl...,78,Television,number of cars,number,image_table,image_table
5,question_6883.json,Does Hilary Duff sing in the movie where T.J. ...,69,Film,yes/no (whether Hilary Duff sings in a specifi...,boolean,table_text,table_text
6,question_2776.json,On the poster for the television show that Cha...,103,Television,time of day (as shown on a poster),time,image_table,image_table
7,question_9578.json,Which film title did Dick Wesson act in first:...,108,Film,film title,creative work,image_table,image_table
8,question_20579.json,"in Team Performance of World Club Challenge, w...",144,Sports,team (club) name,sports team,image_text,image_table
9,question_6113.json,What vehicle is Crash Bandicoot on the cover o...,84,Video games,vehicle (type/name shown on a cover),vehicle,image_table,image_table


In [67]:
labels = sorted(set(multimodal_dataset["modality"]))

print(labels)

results = {}

for label in labels:
    tp = fp = fn = 0
    
    for _, row in multimodal_dataset.iterrows():
        gt = row["modality"]
        pred = row["modality_answers"]
        
        if pred == label and gt == label:
            tp += 1
        elif pred == label and gt != label:
            fp += 1
        elif pred != label and gt == label:
            fn += 1
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = (2*precision*recall/(precision+recall)) if (precision+recall) > 0 else 0
    
    results[label] = {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

['image_table', 'image_text', 'table_table', 'table_text']


In [69]:
pd.DataFrame(results)

,image_table,image_text,table_table,table_text
TP,10.000000,7.000000,5.0,10.000000
FP,1.000000,0.000000,0.0,2.000000
FN,0.000000,3.000000,0.0,0.000000
precision,0.909091,1.000000,1.0,0.833333
recall,1.000000,0.700000,1.0,1.000000
f1,0.952381,0.823529,1.0,0.909091


# Sixth Approach: Unimodal and Multimodal with only one prompt

In [7]:
def dataset_build(dataset):
    image_rows = dataset[dataset["modality"].astype(str) == "['image']"].sample(7, random_state=42)
    text_rows  = dataset[dataset["modality"].astype(str) == "['text']"].sample(7, random_state=42)
    table_rows = dataset[dataset["modality"].astype(str) == "['table']"].sample(7, random_state=42)
    
    image_text_rows = dataset[dataset["modality"].astype(str) == "['image', 'text']"].sample(7, random_state=42)
    text_image_rows = dataset[dataset["modality"].astype(str) == "['text', 'image']"].sample(7, random_state=42)
    image_table_rows  = dataset[dataset["modality"].astype(str) == "['image', 'table']"].sample(7, random_state=42)
    table_image_rows  = dataset[dataset["modality"].astype(str) == "['table', 'image']"].sample(7, random_state=42)
    
    text_table_rows  = dataset[dataset["modality"].astype(str) == "['text', 'table']"].sample(7, random_state=42)
    table_text_rows  = dataset[dataset["modality"].astype(str) == "['table', 'text']"].sample(7, random_state=42)
    
    table_table_rows  = dataset[dataset["modality"].astype(str) == "['table', 'table']"].sample(7, random_state=42)
    
    test_dataset = pd.concat([image_rows, text_rows, table_rows, image_text_rows, text_image_rows, image_table_rows, table_image_rows, text_table_rows, table_text_rows, table_table_rows]) \
                   .sample(frac=1, random_state=42) \
                   .reset_index(drop=False)
    
    return test_dataset

In [8]:
multimodal_unimodal_dataset = dataset_build(dataset)

In [9]:
multimodal_unimodal_dataset["modality_answers"] = [[] for _ in range(len(multimodal_unimodal_dataset))]

In [10]:
class ModalityDecision(BaseModel):
    modalities: Literal[
        "image", 
        "text", 
        "table",
        "image_text",
        "image_table",
        "text_table",
    ] = Field(..., description="Predicted modality combination")

In [11]:
def prompt_all(question, images, paragraphs, table):
    
    images_text = "\n\n".join([
        f"{i+1}. Title: {img['title']}" 
        for i, img in enumerate(images)
    ])

    image_inputs = []
    for img in images:
        image64 = encode_image(os.path.join(FINAL_DATASET_IMAGES, img["path"]))
        image_inputs.append({
            "title": img["title"],
            "image_url": f"data:image/jpeg;base64,{image64}"
        })
        
    paragraphs_text = "\n\n".join([
        f"{i+1}. Title: {p['title']}\nContent: {p['text']}" 
        for i, p in enumerate(paragraphs)
    ])
    
    json_table = json.load(open(os.path.join(TABLE_DIR, table["json"]), "rb"))

    tables_text = f"""
    Title: {json_table["title"]}
    Table name: {json_table["table_name"]}
    Content: {json_table["table"]}
    """

    response = client.responses.parse(
        model="gpt-5.2",
        input=[
            {
                "role": "system",
                "content": """
You are a multimodal modality-selection classifier.

You will be given:
- A question
- IMAGES (titles + image content)
- TEXT paragraphs (titles + content)
- TABLES (titles + table content)

Your job is to decide which modality combination is required to answer the question.

IMPORTANT:
The output must be EXACTLY ONE of the following labels:
- image          (answer can be derived from image alone)
- text           (answer can be derived from text alone)
- table          (answer can be derived from table alone)
- image_text     (answer requires combining IMAGE + TEXT)
- image_table    (answer requires combining IMAGE + TABLE)
- text_table     (answer requires combining TEXT + TABLE)

Rules:
- Use ONLY the provided data.
- Do NOT use outside knowledge or your knowledge.
- Do NOT guess.
- Pick the best matching label.
- Output ONLY the label (no explanation).
"""
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": f"""
Question:
{question["original_question"]}

IMAGES:
{images_text}

TEXT paragraphs:
{paragraphs_text}

TABLES:
{tables_text}

Which modality or modality combination is required? Return only one label.
"""
                    },
                    *[
                        {"type": "input_image", "image_url": img["image_url"]}
                        for img in image_inputs
                    ]
                ]
            }
        ],
        text_format=ModalityDecision,
    )

    return response.output_parsed.modalities

In [12]:
def derive_answers_all_one_prompt(dataset): 
    for i, question in dataset.iterrows(): 
        print(f"Question number: {i}, Question JSON: {question['index']}, Question: {question['original_question']}")
        json_association = json.load(open(os.path.join(ASSOCIATION_DIR, question["index"]), "rb"))
        
        image_set = json_association["image_set"]
        text_set = json_association["text_set"]
        table_set = json_association["table_set"][0]
        
        modality_answer = prompt_all(question, image_set, text_set, table_set)
        
        print("Answer for images: ", modality_answer)
        
        dataset.at[i, "modality_answers"] = modality_answer

In [16]:
derive_answers_all_one_prompt(multimodal_unimodal_dataset)

Question number: 0, Question JSON: question_16282.json, Question: The female artist with long brown hair extending past her shoulders that has the most number-one songs on the German airplay chart sings the song wild thoughts featuring who on the guitar?
Answer for images:  image_table
Question number: 1, Question JSON: question_13547.json, Question: What object is seen in the center of the Days of Our Lives poster?
Answer for images:  image
Question number: 2, Question JSON: question_12447.json, Question: The song "With a Flair" was written for a 1971 Disney musical that was based on the books "The Magic Bedknob; or, How to Become a Witch in Ten Easy Lessons" and "Bonfires and Broomsticks".  What part did Tessie O'Shea play in this movie?
Answer for images:  table
Question number: 3, Question JSON: question_17634.json, Question: Which location with red seating areas is where Ashley Moloney made personal bests?
Answer for images:  image_table
Question number: 4, Question JSON: question

In [17]:
# pk.dump(multimodal_unimodal_dataset, open("multimodal_unimodal_one_prompt.pk", "wb"))

In [20]:
multimodal_unimodal_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_16282.json,The female artist with long brown hair extendi...,188,Music,person (guitarist),person,"[image, text]",image_table
1,question_13547.json,What object is seen in the center of the Days ...,66,Television,object depicted (center of poster),object,[image],image
2,question_12447.json,"The song ""With a Flair"" was written for a 1971...",237,Film,character role (part played by an actor in a m...,role,"[text, table]",table
3,question_17634.json,Which location with red seating areas is where...,82,Sports,location (venue/stadium/track) with red seatin...,location,[image],image_table
4,question_6353.json,What team played the Seattle Seahawks in the 1...,169,Sports,NFL team (opponent team name),sports team,"[text, table]",text_table
...,...,...,...,...,...,...,...,...
65,question_19723.json,in Los Angeles Chargers of List of NFL playoff...,120,Sports,date or year of a logo change,time,"[image, text]",text_table
66,question_21696.json,"In the 2009 FC Rostov season transfers in, wha...",117,Sports,football (soccer) club name,organization,[table],table
67,question_6883.json,Does Hilary Duff sing in the movie where T.J. ...,69,Film,yes/no (whether Hilary Duff sings in a specifi...,boolean,"[table, text]",text_table
68,question_23720.json,Was Toni Polster the Austrian Football Bundesl...,83,Sports,yes/no (boolean),boolean,[table],table


In [18]:
# multimodal_unimodal_dataset1 = pk.load(open("multimodal_unimodal_one_prompt.pk", "rb"))

In [23]:
multimodal_unimodal_dataset["modality"] = multimodal_unimodal_dataset["modality"].apply(lambda x: ["table"] if x == ['table', 'table'] else x)
multimodal_unimodal_dataset["modality"] = multimodal_unimodal_dataset["modality"].apply(normalize_combo)
multimodal_unimodal_dataset["modality_answers"] = multimodal_unimodal_dataset["modality_answers"].apply(normalize_combo)

In [24]:
multimodal_unimodal_dataset

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_16282.json,The female artist with long brown hair extendi...,188,Music,person (guitarist),person,image_text,image_table
1,question_13547.json,What object is seen in the center of the Days ...,66,Television,object depicted (center of poster),object,image,image
2,question_12447.json,"The song ""With a Flair"" was written for a 1971...",237,Film,character role (part played by an actor in a m...,role,table_text,table
3,question_17634.json,Which location with red seating areas is where...,82,Sports,location (venue/stadium/track) with red seatin...,location,image,image_table
4,question_6353.json,What team played the Seattle Seahawks in the 1...,169,Sports,NFL team (opponent team name),sports team,table_text,table_text
...,...,...,...,...,...,...,...,...
65,question_19723.json,in Los Angeles Chargers of List of NFL playoff...,120,Sports,date or year of a logo change,time,image_text,table_text
66,question_21696.json,"In the 2009 FC Rostov season transfers in, wha...",117,Sports,football (soccer) club name,organization,table,table
67,question_6883.json,Does Hilary Duff sing in the movie where T.J. ...,69,Film,yes/no (whether Hilary Duff sings in a specifi...,boolean,table_text,table_text
68,question_23720.json,Was Toni Polster the Austrian Football Bundesl...,83,Sports,yes/no (boolean),boolean,table,table


In [28]:
mask = multimodal_unimodal_dataset.apply(
    lambda row: (row["modality"] != row["modality_answers"]) and
                (len(set(row["modality"].split("_")).intersection(row["modality_answers"].split("_"))) > 0),
    axis=1
)

filtered = multimodal_unimodal_dataset[mask]

In [31]:
filtered

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_16282.json,The female artist with long brown hair extendi...,188,Music,person (guitarist),person,image_text,image_table
2,question_12447.json,"The song ""With a Flair"" was written for a 1971...",237,Film,character role (part played by an actor in a m...,role,table_text,table
3,question_17634.json,Which location with red seating areas is where...,82,Sports,location (venue/stadium/track) with red seatin...,location,image,image_table
7,question_17568.json,In the location out of surrey or london of kin...,116,Other,type of wheel,object type,image_text,image
32,question_3021.json,What is the meaning of the colors of the flag ...,188,Sports,meanings/symbolism of flag colors,description,image_text,table_text
46,question_8874.json,Which video game title was Danielle Nicolet a ...,178,Video games,video game title (the earlier of two compared ...,video game,image_text,image_table
48,question_14856.json,What are the names of the colosseum in the cit...,164,Buildings,names of a colosseum (building name or names),building,image_text,image_table
51,question_2526.json,What date did the first episode of the show st...,92,Television,air date (date of first episode),date,table_text,text
56,question_10125.json,Which artist's song did Amylea sing in the sem...,91,Television,artist (song original performer),person,table_text,table
61,question_11373.json,Among the Argentine football teams that were r...,125,Sports,Argentine football team (club) name,sports team,image,image_table


In [30]:
filtered[filtered["modality"] == "image_text"]

,index,original_question,question_length,question_topic,question_answer_class_specific,question_answer_class_general,modality,modality_answers
0,question_16282.json,The female artist with long brown hair extendi...,188,Music,person (guitarist),person,image_text,image_table
7,question_17568.json,In the location out of surrey or london of kin...,116,Other,type of wheel,object type,image_text,image
32,question_3021.json,What is the meaning of the colors of the flag ...,188,Sports,meanings/symbolism of flag colors,description,image_text,table_text
46,question_8874.json,Which video game title was Danielle Nicolet a ...,178,Video games,video game title (the earlier of two compared ...,video game,image_text,image_table
48,question_14856.json,What are the names of the colosseum in the cit...,164,Buildings,names of a colosseum (building name or names),building,image_text,image_table
63,question_22206.json,Which location won a LEAF Award first: Where ...,175,Film,location (place) that won a LEAF Award first,location,image_text,image_table
65,question_19723.json,in Los Angeles Chargers of List of NFL playoff...,120,Sports,date or year of a logo change,time,image_text,table_text


In [1]:
def calculate_metrics(dataset): 
    labels = sorted(set(dataset["modality"]))
    
    print(labels)
    
    results = {}
    
    for label in labels:
        tp = fp = fn = 0
        
        for _, row in dataset.iterrows():
            gt = row["modality"]
            pred = row["modality_answers"]
            
            if pred == label and gt == label:
                tp += 1
            elif pred == label and gt != label:
                fp += 1
            elif pred != label and gt == label:
                fn += 1
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        results[label] = {
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }
        
    return results

In [131]:
results = calculate_metrics(multimodal_unimodal_dataset)

['image', 'image_table', 'image_text', 'table', 'table_text', 'text']


In [133]:
pd.DataFrame(results)

,image,image_table,image_text,table,table_text,text
TP,5.000000,14.000000,6.000000,14.000000,10.000000,7.000000
FP,1.000000,6.000000,0.000000,3.000000,3.000000,1.000000
FN,2.000000,0.000000,8.000000,0.000000,4.000000,0.000000
precision,0.833333,0.700000,1.000000,0.823529,0.769231,0.875000
recall,0.714286,1.000000,0.428571,1.000000,0.714286,1.000000
f1,0.769231,0.823529,0.600000,0.903226,0.740741,0.933333
